# Task 4 – Hypothesis Testing: Customer Revenue by Gender

## Objective
Test whether mean customer-level revenue differs between female and male customers.

### Hypotheses
- **H₀:** Mean customer-level revenue is equal for female and male customers.
- **H₁:** Mean customer-level revenue differs for female and male customers.
- **Significance level:** α = 0.05
- **Statistical test:** Welch two-sample t-test
- **Confidence interval:** 95%
- **Effect size:** Cohen's d

**Unit of analysis:** one observation per unique customer, using aggregated `Total_Revenue`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Robust dataset discovery
candidates = [
    Path("data/Cleaned_ApexPlanet_Sales_Dataset.xlsx"),
    Path("Cleaned_ApexPlanet_Sales_Dataset(1)(1).xlsx"),
    Path("/mnt/data/Cleaned_ApexPlanet_Sales_Dataset(1)(1).xlsx"),
    Path("/mnt/data/Cleaned_ApexPlanet_Sales_Dataset.xlsx"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Cleaned dataset not found. Update DATA_PATH.")

print("Using dataset:", DATA_PATH)


In [1]:
# Load and validate source data
df = pd.read_excel(DATA_PATH)

print("Shape:", df.shape)
print("Unique orders:", df["Order_ID"].nunique())
print("Unique customers:", df["Customer_ID"].nunique())
print("Total revenue:", round(df["Total_Sales"].sum(), 2))
print("Total units:", int(df["Quantity"].sum()))

required = [
    "Order_ID", "Customer_ID", "Customer_Name", "Gender",
    "City", "Quantity", "Total_Sales", "Order_Date"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Required columns validated successfully.")


Shape: (1000, 15)
Unique orders: 992
Unique customers: 947
Total revenue: 139399439.65
Total units: 5435
Required columns validated successfully.


In [1]:
# Aggregate sales to one row per customer
customer = (
    df.groupby(
        ["Customer_ID", "Customer_Name", "Gender", "City"],
        as_index=False
    )
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Total_Revenue=("Total_Sales", "sum"),
        Total_Units=("Quantity", "sum"),
        Last_Order_Date=("Order_Date", "max"),
    )
)

customer["Last_Order_Date"] = pd.to_datetime(customer["Last_Order_Date"], errors="coerce")
print("Customer-level rows:", len(customer))
customer.head()


Customer-level rows: 947


In [1]:
# Descriptive statistics by gender
gender_summary = (
    customer[customer["Gender"].isin(["Female", "Male"])]
    .groupby("Gender")["Total_Revenue"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)
gender_summary


        count        mean     median         std       min        max
Female    463    144561.77   113543.48    122604.64    1005.21  867333.24
Male      484    149725.91   111517.88    126375.62     437.34  769479.75

In [1]:
# Welch two-sample t-test and 95% confidence interval
female = customer.loc[customer["Gender"] == "Female", "Total_Revenue"].dropna().to_numpy()
male = customer.loc[customer["Gender"] == "Male", "Total_Revenue"].dropna().to_numpy()

alpha = 0.05
test = stats.ttest_ind(female, male, equal_var=False)

n1, n2 = len(female), len(male)
s1_sq, s2_sq = female.var(ddof=1), male.var(ddof=1)

df_welch = (
    (s1_sq / n1 + s2_sq / n2) ** 2
    / (
        ((s1_sq / n1) ** 2) / (n1 - 1)
        + ((s2_sq / n2) ** 2) / (n2 - 1)
    )
)

mean_female = female.mean()
mean_male = male.mean()
mean_diff = mean_female - mean_male

se_diff = np.sqrt(s1_sq / n1 + s2_sq / n2)
t_crit = stats.t.ppf(1 - alpha / 2, df_welch)

ci_low = mean_diff - t_crit * se_diff
ci_high = mean_diff + t_crit * se_diff

pooled_sd = np.sqrt(
    ((n1 - 1) * s1_sq + (n2 - 1) * s2_sq) / (n1 + n2 - 2)
)
cohens_d = mean_diff / pooled_sd

decision = "Fail to reject H0" if test.pvalue >= alpha else "Reject H0"

results = pd.DataFrame({
    "Metric": [
        "Female n", "Male n", "Female mean revenue", "Male mean revenue",
        "Mean difference (Female - Male)", "t-statistic", "Welch df",
        "p-value", "95% CI lower", "95% CI upper", "Cohen's d", "Decision"
    ],
    "Value": [
        n1, n2, mean_female, mean_male, mean_diff, test.statistic, df_welch,
        test.pvalue, ci_low, ci_high, cohens_d, decision
    ]
})

results


Female n = 463
Male n = 484
Female mean revenue = ₹144,561.77
Male mean revenue   = ₹149,725.91
Mean difference (Female - Male) = ₹-5,164.14
t-statistic = -0.6383
Welch df = 944.81
p-value = 0.5235
95% CI = [₹-21,042.49, ₹10,714.21]
Cohen's d = -0.0415
Decision = Fail to reject H0


In [ ]:
# Visual check: group means with 95% confidence intervals
female_se = female.std(ddof=1) / np.sqrt(len(female))
male_se = male.std(ddof=1) / np.sqrt(len(male))

means = [mean_female, mean_male]
errors = [
    stats.t.ppf(0.975, len(female) - 1) * female_se,
    stats.t.ppf(0.975, len(male) - 1) * male_se
]

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(["Female", "Male"], means, yerr=errors, fmt="o", capsize=6, linewidth=2)
ax.set_title("Mean Customer-Level Revenue by Gender (95% CI)")
ax.set_ylabel("Mean Revenue")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Statistical Interpretation

- **Female mean customer revenue:** ₹144,561.77
- **Male mean customer revenue:** ₹149,725.91
- **Observed mean difference (Female − Male):** ₹-5,164.14
- **Welch t-statistic:** -0.6383
- **p-value:** 0.5235
- **95% CI:** [₹-21,042.49, ₹10,714.21]
- **Cohen's d:** -0.0415
- **Decision:** **Fail to reject H₀** at α = 0.05.

### Business conclusion
The sample does not provide sufficient statistical evidence that average customer-level revenue differs by gender. The confidence interval contains zero, and the p-value is above 0.05. This result should not be interpreted as proof that gender can never be associated with revenue; it only means that this specific mean comparison is not statistically significant in this dataset.
